<a href="https://colab.research.google.com/github/harshvardhan60792/word_predictor/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import requests, re, gc, os

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU found  : {bool(tf.config.list_physical_devices('GPU'))}")

# Limit TensorFlow GPU memory growth so it doesn't grab all VRAM at once
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print("GPU memory growth: enabled")

TensorFlow : 2.20.0
GPU found  : True
GPU memory growth: enabled


In [3]:
CORPUS_CHARS = 200_000

def download_corpus(max_chars: int) -> str:
    url = "https://www.gutenberg.org/files/100/100-0.txt"
    print("Downloading Shakespeare corpus …")
    r = requests.get(url, timeout=30)
    r.encoding = "utf-8"
    raw = r.text
    # Strip Gutenberg header / footer
    start = raw.find("THE SONNETS")
    end   = raw.find("End of the Project Gutenberg")
    text  = raw[start:end]
    # Clean: lowercase, keep letters + apostrophes, collapse spaces
    text  = text.lower()
    text  = re.sub(r"[^a-z\s']", "", text)
    text  = re.sub(r"\s+", " ", text).strip()
    sliced = text[:max_chars]
    print(f"Corpus ready: {len(sliced):,} characters")
    return sliced

corpus = download_corpus(CORPUS_CHARS)
print(f"Sample: …{corpus[200:350]}…")

Corpus ready: 200,000 characters
Sample: …of king henry the fourth the second part of king henry the fourth the life of king henry the fifth the first part of henry the sixth the second part o…


In [4]:
MAX_VOCAB   = 10_000   # only the 10 000 most frequent words
SEQ_LENGTH  = 20       # context window: predict word 21 from words 1-20

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts([corpus])

vocab_size   = min(MAX_VOCAB, len(tokenizer.word_index)) + 1
token_list   = tokenizer.texts_to_sequences([corpus])[0]

print(f"Vocabulary size (capped) : {vocab_size:,}")
print(f"Total tokens in corpus   : {len(token_list):,}")

# We no longer need the raw corpus string in RAM — free it
del corpus
gc.collect()
print("Raw corpus freed from RAM ✓")

Vocabulary size (capped) : 5,193
Total tokens in corpus   : 38,806
Raw corpus freed from RAM ✓


In [5]:
BATCH_SIZE = 128

#  Build all sequences as a single numpy array (much cheaper than
#        a list of lists because there's no Python object overhead).
#        Shape: (n_sequences, SEQ_LENGTH + 1)  — last column is the label.

n_sequences = len(token_list) - SEQ_LENGTH
print(f"Building sequence array: {n_sequences:,} sequences × {SEQ_LENGTH+1} tokens")

# Pre-allocate one array — avoids repeated reallocation
sequences = np.zeros((n_sequences, SEQ_LENGTH + 1), dtype=np.int32)
for i in range(n_sequences):
    sequences[i] = token_list[i : i + SEQ_LENGTH + 1]

# Split into features and sparse (integer) labels
X_all = sequences[:, :-1]   # shape (n, SEQ_LENGTH)   int32
y_all = sequences[:, -1]    # shape (n,)               int32  ← NOT one-hot

print(f"X_all shape : {X_all.shape}  ({X_all.nbytes / 1e6:.1f} MB)")
print(f"y_all shape : {y_all.shape}  ({y_all.nbytes / 1e6:.1f} MB)")

# Free the intermediate arrays
del sequences, token_list
gc.collect()
print("Intermediate arrays freed ✓")

# ── 4b: Wrap in tf.data for efficient GPU feeding
BUFFER_SIZE = min(n_sequences, 20_000)   # shuffle buffer (not full dataset)

dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_all, y_all))
    .shuffle(BUFFER_SIZE, seed=42)
    .batch(BATCH_SIZE, drop_remainder=False)
    .prefetch(tf.data.AUTOTUNE)   # overlap CPU pre-processing with GPU compute
)

# Validation split (last 15 % of data)
total_batches = len(X_all) // BATCH_SIZE
val_batches   = max(1, int(total_batches * 0.15))
train_batches = total_batches - val_batches

train_ds = dataset.take(train_batches)
val_ds   = dataset.skip(train_batches)

print(f"\ntf.data pipeline ready")
print(f"  Train batches : {train_batches:,}")
print(f"  Val   batches : {val_batches:,}")
print(f"  Batch size    : {BATCH_SIZE}")

Building sequence array: 38,786 sequences × 21 tokens
X_all shape : (38786, 20)  (3.1 MB)
y_all shape : (38786,)  (0.2 MB)
Intermediate arrays freed ✓

tf.data pipeline ready
  Train batches : 258
  Val   batches : 45
  Batch size    : 128


In [6]:

EMBED_DIM   = 100
LSTM_UNITS1 = 128
LSTM_UNITS2 = 64
DROPOUT     = 0.3

def build_model(vocab_size, seq_length, embed_dim, lstm1, lstm2, dropout):
    model = Sequential([
        # Embedding: integer index → dense float vector
        Embedding(input_dim=vocab_size,
                  output_dim=embed_dim,
                  input_length=seq_length,
                  name="embeddings"),

        # First LSTM: return full sequence so the second LSTM gets context
        LSTM(lstm1, return_sequences=True, name="lstm_1"),
        Dropout(dropout, name="drop_1"),

        # Second LSTM: return only final hidden state
        LSTM(lstm2, return_sequences=False, name="lstm_2"),
        Dropout(dropout, name="drop_2"),

        # Output: probability over every word in the vocabulary
        Dense(vocab_size, activation="softmax", name="output"),
    ], name="LSTM_NextWord_Safe")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        # ↓ This is the critical fix — no one-hot matrix in memory
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model(vocab_size, SEQ_LENGTH,
                    EMBED_DIM, LSTM_UNITS1, LSTM_UNITS2, DROPOUT)
model.summary()

# Quick sanity check: one forward pass shouldn't OOM
test_batch = tf.zeros((2, SEQ_LENGTH), dtype=tf.int32)
_ = model(test_batch, training=False)
print("\nSanity forward pass: OK ✓")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "LSTM_NextWord_Safe"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embeddings (Embedding)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Sanity forward pass: OK ✓


In [7]:

callbacks = [
    # Stop if val_loss doesn't improve for 4 epochs; restore best weights
    EarlyStopping(monitor="val_loss", patience=4,
                  restore_best_weights=True, verbose=1),
    # Halve learning rate if val_loss stalls for 2 epochs
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=2, min_lr=1e-6, verbose=1),
]

print("Starting training …  (EarlyStopping will handle epoch count)\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,              # EarlyStopping will stop before 50 in practice
    callbacks=callbacks,
    verbose=1,
)

# ── Results summary
best_epoch   = np.argmin(history.history["val_loss"])
best_val_loss = history.history["val_loss"][best_epoch]
best_val_acc  = history.history["val_accuracy"][best_epoch]
perplexity    = np.exp(best_val_loss)

print(f"\n{'─'*45}")
print(f"  Best epoch         : {best_epoch + 1}")
print(f"  Val loss           : {best_val_loss:.4f}")
print(f"  Val accuracy       : {best_val_acc:.4f}")
print(f"  Perplexity         : {perplexity:.2f}")
print(f"{'─'*45}")

# Save model
model.save("lstm_next_word_safe.h5")
print("Model saved → lstm_next_word_safe.h5")


Starting training …  (EarlyStopping will handle epoch count)

Epoch 1/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.0275 - loss: 6.9272 - val_accuracy: 0.0340 - val_loss: 6.5587 - learning_rate: 0.0010
Epoch 2/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.0283 - loss: 6.5979 - val_accuracy: 0.0288 - val_loss: 6.4423 - learning_rate: 0.0010
Epoch 3/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.0288 - loss: 6.5034 - val_accuracy: 0.0331 - val_loss: 6.3965 - learning_rate: 0.0010
Epoch 4/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.0320 - loss: 6.4479 - val_accuracy: 0.0384 - val_loss: 6.3138 - learning_rate: 0.0010
Epoch 5/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.0363 - loss: 6.4039 - val_accuracy: 0.0396 - val_loss: 6.2453 - learning_rate: 0.0010
Epoch 6/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.0394 - loss: 6.3495 - val_accuracy: 0.0371 - val_loss: 6.2383 - learning_rate: 0.0010
Epoch 7/50
258/258 ━


─────────────────────────────────────────────
  Best epoch         : 45
  Val loss           : 4.9273
  Val accuracy       : 0.1222
  Perplexity         : 138.01
─────────────────────────────────────────────
Model saved → lstm_next_word_safe.h5


In [8]:
def sample_with_temperature(probs: np.ndarray, temperature: float) -> int:
    """Draw a sample from a probability array with temperature scaling."""
    probs = probs.astype(np.float64)
    # Subtract max before exp for numerical stability (log-sum-exp trick)
    logits = np.log(probs + 1e-10) / temperature
    logits -= logits.max()
    exp_logits = np.exp(logits)
    scaled = exp_logits / exp_logits.sum()
    return int(np.random.choice(len(scaled), p=scaled))

# Build reverse lookup: index → word  (built once, reused)
index_to_word = {idx: word for word, idx in tokenizer.word_index.items()
                 if idx < vocab_size}

def generate_text(seed: str, n_words: int, temperature: float = 0.8) -> str:
    """
    Autoregressive generation:
      1. Tokenise seed text
      2. Predict next word
      3. Append predicted word to context
      4. Repeat n_words times
    """
    generated = seed.lower().strip()

    for _ in range(n_words):
        # Tokenise current context (unknown words → OOV index)
        tokens = tokenizer.texts_to_sequences([generated])[0]
        # Pad / truncate to SEQ_LENGTH
        tokens = pad_sequences([tokens], maxlen=SEQ_LENGTH, padding="pre")
        # Forward pass → (1, vocab_size) probability vector
        probs = model.predict(tokens, verbose=0)[0]
        # Sample with temperature
        idx = sample_with_temperature(probs, temperature)
        word = index_to_word.get(idx, "<OOV>")
        generated += " " + word

    return generated

# ── Run three generations at different temperatures
seed = "to be or not to be"

print("=" * 55)
print(f"Seed: \"{seed}\"\n")

for temp, label in [(0.3, "Conservative (temp=0.3)"),
                    (0.8, "Balanced    (temp=0.8)"),
                    (1.2, "Creative    (temp=1.2)")]:
    print(f"── {label}")
    print(generate_text(seed, n_words=30, temperature=temp))
    print()

Seed: "to be or not to be"

── Conservative (temp=0.3)
to be or not to be not far to have you to be no eyes that i am not i have not to be not is my love and i have be i have not i

── Balanced    (temp=0.8)
to be or not to be did say i mean thus away to his summer countess i am i demand and all the side of the bloods of burn he are admirable i have i love

── Creative    (temp=1.2)
to be or not to be since mine seeing for not one he making all perjured altogether of the interchange a english music when thou art defaced a merit and my place madam been beware of

